In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType
from pyspark.sql.functions import col, to_timestamp, from_utc_timestamp, date_format, round, lit
from datetime import datetime

In [0]:
jdbc_url = "jdbc:sqlserver://capstone-database-server.database.windows.net:1433;database=writedatabasesilverlayer;"
connection_properties = {
    "user": "capstonedioxieteam",
    "password": "Connhenbeo1@",
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

old_data = spark.read.jdbc(
    url=jdbc_url,
    table="Bronze.Company_Information",
    properties=connection_properties
)

In [0]:
cleaned_data = old_data.dropDuplicates()
cleaned_data = cleaned_data.fillna(0)

In [0]:
cleaned_data = cleaned_data.withColumn("updated_date", lit(datetime.today().date()))
display(cleaned_data)

In [0]:
cleaned_data.toPandas().to_csv('/dbfs/FileStore/Silver/Company_Information_Silver.csv', index=False)

jdbc_url = "jdbc:sqlserver://capstone-database-server.database.windows.net:1433;database=writedatabasesilverlayer;"
connection_properties = {
    "user": "capstonedioxieteam",
    "password": "Connhenbeo1@",
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

In [0]:
df_spark_Company_Information_silver = spark.read.csv(
    "/FileStore/Silver/Company_Information_Silver.csv",
    header=True,
    inferSchema=True
)

df_spark_Company_Information_silver.write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "Silver.Company_Information") \
    .option("user", connection_properties["user"]) \
    .option("password", connection_properties["password"]) \
    .option("driver", connection_properties["driver"]) \
    .mode("overwrite") \
    .option("batchsize", 10000) \
    .option("numPartitions", 8) \
    .save()

print("Data successfully written to Azure SQL Database.")